
# Cinemática Directa: El Modelo Espacial del Robot 🦾

<a href="https://colab.research.google.com/github/Reve7339/robotics-foundations/blob/main/Direct_Kinematics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

¡Bienvenido al segundo módulo! Mientras que en el primero aprendimos a rotar objetos, aquí daremos el paso definitivo: **posicionar y orientar un brazo robótico completo en el espacio tridimensional**.
Para lograrlo, pasaremos de las limitadas matrices $3\times3$ a las poderosas **Transformaciones Homogéneas** ($4\times4$) y aprenderemos el estándar de oro de la industria: la **Convención Denavit-Hartenberg (DH)**.

Ejecuta la siguiente celda para cargar el motor matemático y gráfico:


In [67]:
%pip install numpy scipy plotly -q
import numpy as np
import plotly.graph_objects as go
print("✅ Motor Cinemático Iniciado.")


Note: you may need to restart the kernel to use updated packages.
✅ Motor Cinemático Iniciado.



---
## 1. Transformaciones Homogéneas

Si queremos describir el estado cinemático del efector final, resulta insuficiente especificar únicamente su orientación relativa (rotación); es imprescindible determinar su vector de posición (traslación) con respecto al sistema de referencia base.
Las matemáticas clásicas obligarían a hacer esto:
$$ p_0 = R_1^0 p_1 + d_1^0 $$

Sin embargo, computar rotaciones y traslaciones mediante operaciones independientes (multiplicación y suma) resulta computacionalmente ineficiente y algebraicamente complejo al evaluar múltiples eslabones en cadena. La genialidad de las **Matrices Homogéneas** radica en empaquetar ambas operaciones en una única matriz de $4\times4$ agregando un $1$ ficticio (coordenada homogénea) a los vectores espaciales.

La matriz de transformación $T$ se define como:
$$ 
T = \begin{bmatrix} R_{3\times3} & d_{3\times1} \\ 0_{1\times3} & 1 \end{bmatrix}
$$
Bajo esta formulación, las operaciones de rotación y traslación quedan unificadas en un único producto matricial iterativo.


In [56]:

def rot_z(theta_deg):
    t = np.radians(theta_deg)
    return np.array([[np.cos(t), -np.sin(t), 0],
                     [np.sin(t),  np.cos(t), 0],
                     [        0,          0, 1]])

# Construyendo una Matriz Homogénea en Python
R = rot_z(45)              # Rotamos 45 grados
d = np.array([[2, 3, 1]]).T  # Nos trasladamos X=2, Y=3, Z=1

# Ensamblaje de la matriz 4x4
T = np.block([
    [R,               d],
    [np.zeros((1,3)), 1]
])

print("Matriz Homogénea Resultante T:\n", np.round(T, 2))


Matriz Homogénea Resultante T:
 [[ 0.71 -0.71  0.    2.  ]
 [ 0.71  0.71  0.    3.  ]
 [ 0.    0.    1.    1.  ]
 [ 0.    0.    0.    1.  ]]



---
## 2. La Convención Denavit-Hartenberg (DH)

En manipuladores con múltiples grados de libertad, asignar sistemas de referencia de manera arbitraria carece de generalidad y dificulta el análisis cinemático sistemático. Jacques Denavit y Richard Hartenberg desarrollaron un método algorítmico estandarizado para asignar ejes rígidamente a cada articulación utilizando **solo 4 parámetros** para determinar la transformación del marco coordenado $i-1$ al marco $i$:

1. $a_i$ (Longitud del eslabón): Distancia en X entre los ejes Z.
2. $\alpha_i$ (Torsión del eslabón): Ángulo en X entre los ejes Z.
3. $d_i$ (Desplazamiento articular): Distancia en Z entre los ejes X. (Variable para prismáticos).
4. $\theta_i$ (Ángulo articular): Ángulo en Z entre los ejes X. (Variable para revolutas).

La secuencia matemática exacta para transformar del marco $(i-1)$ al $(i)$ es:
$$ A_i^{i-1} = Rot_z(\theta_i) \cdot Trans_z(d_i) \cdot Trans_x(a_i) \cdot Rot_x(\alpha_i) $$

Al multiplicar estas cuatro matrices matrices homogéneas elementales, obtenemos la **Matriz General de Denavit-Hartenberg**:

$$
A_i^{i-1} = \begin{bmatrix}
\cos\theta_i & -\sin\theta_i\cos\alpha_i & \sin\theta_i\sin\alpha_i & a_i\cos\theta_i \\
\sin\theta_i & \cos\theta_i\cos\alpha_i & -\cos\theta_i\sin\alpha_i & a_i\sin\theta_i \\
0 & \sin\alpha_i & \cos\alpha_i & d_i \\
0 & 0 & 0 & 1
\end{bmatrix}
$$


In [57]:

def matriz_dh(a, alpha_deg, d, theta_deg):
    '''Genera la matriz de transformación homogénea de Denavit-Hartenberg.'''
    alpha = np.radians(alpha_deg)
    theta = np.radians(theta_deg)
    
    ct = np.cos(theta); st = np.sin(theta)
    ca = np.cos(alpha); sa = np.sin(alpha)
    
    A = np.array([
        [ct, -st*ca,  st*sa, a*ct],
        [st,  ct*ca, -ct*sa, a*st],
        [ 0,     sa,     ca,    d],
        [ 0,      0,      0,    1]
    ])
    return A

# Ejemplo: Matriz para un eslabón de longitud a=10 y rotado 30°
print("A_1^0 =\n", np.round(matriz_dh(a=10, alpha_deg=0, d=0, theta_deg=30), 2))


A_1^0 =
 [[ 0.87 -0.5   0.    8.66]
 [ 0.5   0.87 -0.    5.  ]
 [ 0.    0.    1.    0.  ]
 [ 0.    0.    0.    1.  ]]



---
## 3. La Cinemática Directa de la Cadena Completa

Para saber exactamente dónde está la punta del robot (el efector final) con respecto al piso (la base $0$), simplemente multiplicamos en orden matricial todas las matrices $A$ de cada eslabón:

$$ T_n^0(q) = A_1^0(q_1) \cdot A_2^1(q_2) \cdot A_3^2(q_3) \dots A_n^{n-1}(q_n) $$

El vector de posición extraído de $T_n^0$ nos dará el $(X, Y, Z)$ exacto de la herramienta.
¡A continuación, entraremos a la Sección 4 donde crearemos el simulador 3D y simularemos robots reales!



---
## 4. [Casos Prácticos] Modelado de Arquitecturas Industriales Clásicas

A continuación, aplicaremos la formulación matemática de Denavit-Hartenberg para simular cuatro arquitecturas cinemáticas fundamentales en la robótica industrial. El modelo se evalúa sistemáticamente al definir los parámetros DH estructurales y operar sobre el vector de variables articulares ($q_i$).

Procedemos a definir la función de renderizado `plot_robot_dh`. Esta rutina gráfica no solo traza la conectividad de los eslabones, sino que proyecta los **sistemas de referencia ortonormales en cada articulación** (X=Rojo, Y=Verde, Z=Azul), permitiendo una inspección visual rigurosa de las transformaciones DH.


In [58]:

def plot_robot_dh(dh_table, name="Robot"):
    '''
    dh_table: lista de diccionarios [{'a':.., 'alpha':.., 'd':.., 'theta':.., 'type': 'R' o 'P'}]
    '''
    T_total = np.eye(4)
    px_prev, py_prev, pz_prev = 0, 0, 0
    
    fig = go.Figure()
    
    # Base
    fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode='markers',
                               marker=dict(size=12, color='black', symbol='square'),
                               name='Base (0)'))
                               
    for i, link in enumerate(dh_table):
        A = matriz_dh(link['a'], link['alpha'], link['d'], link['theta'])
        T_total = T_total @ A
        
        px, py, pz = T_total[0:3, 3]
        
        # Tipo de articulación determina cómo se dibuja el "hueso"
        j_type = link.get('type', 'R')
        if j_type == 'R':
            line_color = 'orange'
            line_width = 12
            marker_symbol = 'circle'
            name_label = f"Eslabón {i+1} (R)"
        else:
            line_color = 'silver'
            line_width = 16
            marker_symbol = 'square'
            name_label = f"Eslabón {i+1} (P - Telescópico)"
            
        # Dibujar el eslabón desde el frame i-1 al frame i
        fig.add_trace(go.Scatter3d(x=[px_prev, px], y=[py_prev, py], z=[pz_prev, pz], 
                                   mode='lines+markers',
                                   line=dict(color=line_color, width=line_width),
                                   marker=dict(size=8, color='black', symbol=marker_symbol),
                                   name=name_label))
                                   
        px_prev, py_prev, pz_prev = px, py, pz
        
        # Dibujar sistema de coordenadas en la articulación
        R = T_total[0:3, 0:3]
        s = 0.5 # escala de las flechas
        # X (Rojo)
        fig.add_trace(go.Scatter3d(x=[px, px+s*R[0,0]], y=[py, py+s*R[1,0]], z=[pz, pz+s*R[2,0]],
                                   mode='lines', line=dict(color='red', width=5), showlegend=False))
        # Y (Verde)
        fig.add_trace(go.Scatter3d(x=[px, px+s*R[0,1]], y=[py, py+s*R[1,1]], z=[pz, pz+s*R[2,1]],
                                   mode='lines', line=dict(color='green', width=5), showlegend=False))
        # Z (Azul) - ¡Es el eje de rotación/traslación!
        fig.add_trace(go.Scatter3d(x=[px, px+s*R[0,2]], y=[py, py+s*R[1,2]], z=[pz, pz+s*R[2,2]],
                                   mode='lines', line=dict(color='blue', width=8), showlegend=False))

    fig.update_layout(title=f"Simulación 3D: {name} <br><sup>(Ejes: Rojo=X, Verde=Y, Azul=Z) | Naranja=Rotación, Plata=Prismático</sup>",
                      scene=dict(xaxis=dict(range=[-3,3]), yaxis=dict(range=[-3,3]), zaxis=dict(range=[0,4]), aspectmode='cube'))
    fig.show()
    return T_total



### 4.1 El Brazo Antropomórfico (Articulado) - RRR
Es la arquitectura que más se parece a un brazo humano (cintura, hombro, codo). Es el estándar para soldadura (KUKA, FANUC).
- **Articulaciones:** 3 de Rotación (Revolutas).


In [59]:

# Variables articulares (Juega cambiándolas)
q1, q2, q3 = 45, 30, -45  # Cintura, Hombro, Codo

dh_antropomorfico = [
    {'a': 0, 'alpha': 90, 'd': 1.0, 'theta': q1, 'type': 'R'},   # Link 1: Sube en Z y gira el eje Z a horizontal
    {'a': 1.5, 'alpha': 0, 'd': 0, 'theta': q2, 'type': 'R'},    # Link 2: Brazo
    {'a': 1.2, 'alpha': 0, 'd': 0, 'theta': q3, 'type': 'R'}     # Link 3: Antebrazo
]

plot_robot_dh(dh_antropomorfico, "Brazo Antropomórfico")


array([[ 6.83012702e-01,  1.83012702e-01,  7.07106781e-01,
         1.73817390e+00],
       [ 6.83012702e-01,  1.83012702e-01, -7.07106781e-01,
         1.73817390e+00],
       [-2.58819045e-01,  9.65925826e-01,  6.12323400e-17,
         1.43941715e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00]])


### 4.2 El Robot Esférico (Stanford Arm) - RRP
Utilizado históricamente y perfecto para alcanzar objetos dentro de cavidades.
- **Articulaciones:** 2 de Rotación (base y elevación) y 1 Prismática (extensión telescópica).


In [60]:

# Variables articulares
q1_rot, q2_rot, q3_dist = 60, 45, 2.0  # Base, Elevación, Extensión de la lanza

dh_esferico = [
    {'a': 0, 'alpha': -90, 'd': 1.0, 'theta': q1_rot, 'type': 'R'},  
    {'a': 0, 'alpha': 90,  'd': 0,   'theta': q2_rot, 'type': 'R'},  
    {'a': 0, 'alpha': 0,   'd': q3_dist, 'theta': 0, 'type': 'P'}    # Eslabón prismático (d varía)
]

plot_robot_dh(dh_esferico, "Robot Esférico")


array([[ 3.53553391e-01, -8.66025404e-01,  3.53553391e-01,
         7.07106781e-01],
       [ 6.12372436e-01,  5.00000000e-01,  6.12372436e-01,
         1.22474487e+00],
       [-7.07106781e-01,  1.79345371e-17,  7.07106781e-01,
         2.41421356e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00]])


### 4.3 El Robot Cilíndrico - RPP
Su área de trabajo forma un cilindro perfecto. Excelente para ensamblaje vertical y paletizado rígido.
- **Articulaciones:** 1 de Rotación (base), 1 Prismática (elevación vertical), 1 Prismática (alcance horizontal).


In [61]:

# Variables articulares
q1_rot, q2_alt, q3_dist = -30, 1.5, 2.0  # Base, Altura en Z, Alcance radial

dh_cilindrico = [
    {'a': 0, 'alpha': 0,   'd': 0.5, 'theta': q1_rot, 'type': 'R'}, # Cintura
    {'a': 0, 'alpha': -90, 'd': q2_alt, 'theta': 0, 'type': 'P'},   # Elevación vertical (Z sube)
    {'a': 0, 'alpha': 0,   'd': q3_dist, 'theta': 0, 'type': 'P'}   # Alcance horizontal
]

plot_robot_dh(dh_cilindrico, "Robot Cilíndrico")


array([[ 8.66025404e-01,  3.06161700e-17,  5.00000000e-01,
         1.00000000e+00],
       [-5.00000000e-01,  5.30287619e-17,  8.66025404e-01,
         1.73205081e+00],
       [ 0.00000000e+00, -1.00000000e+00,  6.12323400e-17,
         2.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00]])


### 4.4 El Robot SCARA - RRPR
Arquitectura predominante en líneas de ensamblaje (Pick and Place). Su diseño, con ejes de revolución paralelos, le otorga alta rigidez estructural en el plano vertical (compliance selectiva) garantizando trayectorias planas a alta velocidad de ejecución.
- **Articulaciones:** 2 de Revolución (movimiento planar), 1 Prismática (descenso vertical), y 1 de Revolución (orientación final).


In [62]:

# Variables articulares
q1, q2, q3_baja, q4_herramienta = 45, -60, 0.5, 0  

dh_scara = [
    {'a': 0,   'alpha': 0,   'd': 2.0, 'theta': 0,  'type': 'R'}, # Pilar fijo (Base al Hombro)
    {'a': 1.5, 'alpha': 0,   'd': 0,   'theta': q1, 'type': 'R'}, # Hombro horizontal (O1 en el codo)
    {'a': 1.0, 'alpha': 180, 'd': 0,   'theta': q2, 'type': 'R'}, # Codo horizontal (invierte Z hacia abajo)
    {'a': 0,   'alpha': 0,   'd': q3_baja, 'theta': 0, 'type': 'P'}, # Baja la herramienta (Prismática)
    {'a': 0,   'alpha': 0,   'd': 0,   'theta': q4_herramienta, 'type': 'R'} # Rota la herramienta
]

plot_robot_dh(dh_scara, "Robot SCARA")


array([[ 9.65925826e-01, -2.58819045e-01, -3.16961915e-17,
         2.02658600e+00],
       [-2.58819045e-01, -9.65925826e-01, -1.18291797e-16,
         8.01841127e-01],
       [ 0.00000000e+00,  1.22464680e-16, -1.00000000e+00,
         1.50000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00]])


---
## 5. Ejercicios Prácticos (Resolución Analítica)

**Instrucciones:** Resuelve los siguientes problemas a mano en tu cuaderno, multiplicando matricialmente las transformaciones homogéneas. Al terminar, ejecuta la celda de código debajo de cada uno para que el simulador valide matemáticamente tus resultados analíticos.

### Ejercicio 1: El Manipulador Planar 2R
Un brazo robótico simple consta de dos eslabones moviéndose sobre el plano XY (como un brazo apoyado sobre una mesa). 
- **Eslabón 1 (Brazo):** Longitud $L_1 = 2\text{ m}$.
- **Eslabón 2 (Antebrazo):** Longitud $L_2 = 1\text{ m}$.
Si los motores giran a $q_1 = 90^\circ$ y $q_2 = -90^\circ$, encuentra analíticamente la matriz de transformación homogénea total $T_2^0$ y extrae la coordenada cartesiana $(X, Y)$ del efector final.
**Pista:** Multiplica a mano las matrices $A_1^0(q_1) \cdot A_2^1(q_2)$. Nota que en un manipulador planar, $d_1=0, d_2=0, \alpha_1=0, \alpha_2=0$.


In [68]:

# Código de comprobación (Ejercicio 1)
# 1. Definimos los parámetros evaluados
A1 = matriz_dh(a=2.0, alpha_deg=0, d=0, theta_deg=90)
A2 = matriz_dh(a=1.0, alpha_deg=0, d=0, theta_deg=-90)

# 2. Multiplicación matricial de la cadena cinemática
T_total = A1 @ A2

print("Matriz Homogénea T_2^0 (Revisa tu cálculo manual):\n")
print(np.round(T_total, 2))
print(f"\nCoordenada Final Alcanzada (X, Y): ({np.round(T_total[0,3],2)}, {np.round(T_total[1,3],2)})")


Matriz Homogénea T_2^0 (Revisa tu cálculo manual):

[[1. 0. 0. 1.]
 [0. 1. 0. 2.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]

Coordenada Final Alcanzada (X, Y): (1.0, 2.0)



### Ejercicio 2: El Robot Cartesiano (Impresora 3D)
Imagina el mecanismo de una impresora 3D compuesto por 3 ejes deslizantes ortogonales (3 articulaciones prismáticas). Suponiendo que la base inercial coincide con la esquina inferior, y se ha introducido el comando $q_x = 3, q_y = 2, q_z = 5$. Construye a mano la matriz $T_3^0$ asumiendo que no hay rotaciones relativas entre los ejes ($\alpha_i=0, \theta_i=0$). 
**Pista:** Escribe directamente una matriz de $4\times4$ que sea la matriz identidad en la submatriz de rotación $R_{3\times3}$ y que contenga el vector de traslación consolidado en su última columna.


In [64]:

# Código de comprobación (Ejercicio 2)
# Traslaciones puras impuestas por los prismas
qx, qy, qz = 3.0, 2.0, 5.0

T_impresora = np.eye(4)
T_impresora[0:3, 3] = [qx, qy, qz]

print("Matriz Homogénea T_3^0 del Robot Cartesiano PPP:\n")
print(np.round(T_impresora, 2))


Matriz Homogénea T_3^0 del Robot Cartesiano PPP:

[[1. 0. 0. 3.]
 [0. 1. 0. 2.]
 [0. 0. 1. 5.]
 [0. 0. 0. 1.]]



### Ejercicio 3: Cinemática Inversa Intuitiva del SCARA
Un robot SCARA con eslabones horizontales de longitudes $L_1 = 2\text{ m}$ y $L_2 = 1\text{ m}$ debe alcanzar una pieza en la coordenada cartesiana $(X=3, Y=0)$.
Asumiendo que la altura estructural del pilar base es de $2\text{ m}$ y la herramienta prismática debe tocar el piso ($Z=0$), resuelve analíticamente a lápiz y papel los valores requeridos para las variables articulares: $q_1, q_2, q_3$.
**Pista:** Usa razonamiento trigonométrico básico. Si la máxima extensión radial es $L_1 + L_2 = 3$, ¿cómo deben estar alineados geométricamente los eslabones para llegar a $X=3$? Si el pilar arranca a $Z=2$, ¿cuánto debe descender $q_3$ para que la punta llegue al piso?


In [65]:

# Código de comprobación (Ejercicio 3)
# Solución teórica esperada obtenida a mano:
# q1 = 0 (Totalmente alineado con el eje X)
# q2 = 0 (Totalmente estirado, sin codo doblado)
# q3 = 2.0 (Desciende la longitud exacta del pilar)

dh_scara_ejercicio = [
    {'a': 0,   'alpha': 0,   'd': 2.0, 'theta': 0,  'type': 'R'}, # Pilar
    {'a': 2.0, 'alpha': 0,   'd': 0,   'theta': 0,  'type': 'R'}, # Hombro (L1) -> q1=0
    {'a': 1.0, 'alpha': 180, 'd': 0,   'theta': 0,  'type': 'R'}, # Codo (L2) -> q2=0
    {'a': 0,   'alpha': 0,   'd': 2.0, 'theta': 0,  'type': 'P'}, # Prisma de descenso -> q3=2.0
    {'a': 0,   'alpha': 0,   'd': 0,   'theta': 0,  'type': 'R'}  # Rotación herramienta
]

T_total_scara = np.eye(4)
for link in dh_scara_ejercicio:
    A = matriz_dh(link['a'], link['alpha'], link['d'], link['theta'])
    T_total_scara = T_total_scara @ A

print("Posición Final validada por el evaluador DH:\n")
print(f"X: {np.round(T_total_scara[0,3], 2)}")
print(f"Y: {np.round(T_total_scara[1,3], 2)}")
print(f"Z: {np.round(T_total_scara[2,3], 2)}")
print("\n¡Si llegaste a las coordenadas (3, 0, 0) a mano con q1=0, q2=0, q3=2, tu razonamiento espacial es excelente!")


Posición Final validada por el evaluador DH:

X: 3.0
Y: -0.0
Z: 0.0

¡Si llegaste a las coordenadas (3, 0, 0) a mano con q1=0, q2=0, q3=2, tu razonamiento espacial es excelente!
